# ETFs: Holdout Predictions

**Chapter 20 - Out-of-sample evaluation**

Every number in this case study so far was measured on the validation folds, and every
choice was made by looking at them: which model family, which configuration, how many
funds to hold, how to size them, which risk control to overlay, what to charge. A result
selected that way cannot also be evidence that the selection was sound - the ranking and
the evidence would be the same measurement.

The holdout is the window nothing has been selected on. This notebook fits the selected
configuration on the history available before that window opens and writes its
predictions over it. [`19_holdout_backtest`](19_holdout_backtest.ipynb) turns those
predictions into a return series with the sizing and the overlay the case study settled
on, and [`20_strategy_analysis`](20_strategy_analysis.ipynb) reads both back.

**What this notebook is careful about**

A holdout prediction is not the validation model scored on a later window. Section 2
fits again, over a training interval that ends before the window opens, and the new
training identity is what makes the refit visible rather than asserted: the identity
covers the CV interval, so a run that came back with the validation training hash would
mean no refit happened. The check is in section 3 and it raises.

**Prerequisites:** [`17_costs`](17_costs.ipynb), which is the last stage that selects.

**Scope:** one training run and one prediction set. No backtest, no selection, no
comparison - those are 19 and 20.

<!-- ML4T_ZH_BEGIN c0000-p000 -->
# ETF：保留測試集預測

**第 20 章——樣本外評估**

本案例研究迄今所有數值都在驗證折上衡量，所有選擇也都是查看驗證折後做出的：選哪個模型家族、哪個組態、持有多少檔基金、如何配置部位、套用哪種風險控制，以及計入多少成本。用這種方式選出的結果，不能同時拿來證明選擇正確，因為排名依據與證據會是同一次衡量。

保留測試集是一段未曾用來做任何選擇的期間。本 Notebook 會以該期間開始前可用的歷史資料，擬合選定的組態，並產生該期間的預測。[`19_holdout_backtest`](19_holdout_backtest.ipynb) 會依案例研究選定的部位規模設定與覆蓋規則，將這些預測轉為報酬率序列；[`20_strategy_analysis`](20_strategy_analysis.ipynb) 則會讀回兩者。

**本 Notebook 特別留意的事項**

保留測試集預測，不是拿在驗證期間擬合的模型去評分後續期間。第 2 節會在保留測試期間開始前結束的訓練區間重新擬合；新的訓練識別項讓重新擬合成為可驗證的事實，而不只是宣稱：識別項涵蓋交叉驗證區間，因此若某次執行傳回驗證階段的訓練雜湊值，就代表沒有重新擬合。第 3 節會檢查此事，並在不符時拋出錯誤。

**先備條件：**[`17_costs`](17_costs.ipynb)，這是最後一個進行篩選的階段。

**範圍：**一次訓練執行與一組預測。不進行回測、篩選或比較；這些工作由第 19 與 20 份 Notebook 負責。
<!-- ML4T_ZH_END c0000-p000 -->


In [ ]:
"""ETFs: Holdout Predictions."""

import polars as pl

from case_studies.research import open_study
from case_studies.research.holdout import build_holdout_training_spec
from case_studies.research.models import reconstruct_locked_model_request
from case_studies.utils.registry import training_hash_from_spec
from case_studies.utils.strategy_analysis import (
    holdout_generations_to_retire,
    registered_holdout_generations,
    resolve_solvent_carrier,
)
from utils.paths import get_case_study_dir

In [ ]:
CASE_STUDY_ID = "etfs"
EXECUTION_TIER = "canonical"
WORKSPACE: str = ""

In [ ]:
study = open_study(CASE_STUDY_ID, execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
CASE_DIR = get_case_study_dir(CASE_STUDY_ID)

## 1. Which configuration the holdout runs

The holdout runs the configuration the case study reports, resolved through the same
`resolve_solvent_carrier` [`17_costs`](17_costs.ipynb) prices. Resolving it again here
rather than passing it along is deliberate: the two notebooks must agree by construction,
and a hash written down in one and read in the other agrees only until the sweep is
rebuilt.

Nothing about the holdout enters this choice. The selected configuration is the cross-stage
validation rank-1 - which for this case study is a risk-overlay run, not the allocation leader -
and it was fixed before this notebook ran.

<!-- ML4T_ZH_BEGIN c0004-p000 -->
## 1. 保留測試集使用哪個組態

保留測試集使用案例研究所報告的組態，透過同一個 `resolve_solvent_carrier` 來解析；[`17_costs`](17_costs.ipynb) 也用它進行計價。此處重新解析，而不直接傳遞，是刻意的安排：兩份 Notebook 必須在設計上取得相同結果；若在一份中寫下雜湊值，再由另一份讀取，只有在掃描未重建前才會一致。

保留測試集的任何資訊都不參與這項選擇。選定的組態是跨階段驗證排名第一者；在本案例研究中，它是風險覆蓋規則的執行結果，而不是配置階段的領先者。這項選擇早在本 Notebook 執行前就已固定。
<!-- ML4T_ZH_END c0004-p000 -->


In [ ]:
carrier = resolve_solvent_carrier(CASE_STUDY_ID)
print(
    f"Selected configuration: {carrier['val_backtest_hash']}  stage={carrier['val_stage']}  "
    f"family={carrier['family']}  config={carrier['config_name']}  "
    f"label={carrier['label']}"
)
print(
    f"  validation Sharpe {carrier['val_sharpe']:.3f}, max drawdown {carrier['max_drawdown']:.3f}"
)
print(f"  fitted by training run {carrier['training_hash']}")

The checkpoint is part of the configuration. Where a family publishes a prediction set per
checkpoint on a declared schedule, the selected configuration's prediction set names one of them,
and refitting without it would produce a model at the end of training rather than the one that
was ranked. A family with no checkpoint dimension stores NULL in both columns and carries that
NULL through unchanged.

<!-- ML4T_ZH_BEGIN c0006-p000 -->
檢查點是組態的一部分。若某模型家族依預先指定的時程，為每個檢查點發布一組預測，選定組態的預測組合就會指明其中一個檢查點；重新擬合時若未保留它，得到的會是訓練結束時的模型，而不是先前參與排名的模型。沒有檢查點維度的模型家族，會在兩個欄位都儲存 NULL，並原樣沿用該 NULL。
<!-- ML4T_ZH_END c0006-p000 -->


In [ ]:
validation_prediction = study.results.open(carrier["val_prediction_hash"])
prediction_record = validation_prediction.registry_record()
CHECKPOINT_KIND = prediction_record["checkpoint_kind"]
CHECKPOINT_VALUE = prediction_record["checkpoint_value"]
print(f"Checkpoint: {CHECKPOINT_KIND}={CHECKPOINT_VALUE}")

## 2. The window, and the model that is allowed to see it

The holdout window is not a choice made here. It is `evaluation.holdout_start` and
`evaluation.holdout_end` from this case study's own `setup.yaml`, read through the same
`canonical_window` the fold derivation and the backtest slice both go through, so the three
cannot disagree.

The training interval is everything available before that window, bounded above by a label
buffer. **For this case study the buffer is doing real work.** The label is a 21-day forward
return, so a row dated `t` records an outcome that is not known until `t + 21` sessions - and
a training set running to the day the window opens would be fitted on labels whose returns
resolve inside it. The buffer is therefore a horizon's worth rather than a formality, and the
derivation refuses to default it: a zero gap here would be a leak.

Everything else about the configuration is carried across unchanged, and the fields that
cannot be - the eligibility manifest, and any parameter this family resolves from a fold's
own training rows - are recomputed against the holdout fold. Carrying those forward would fit
a model keyed to the validation folds and call it a retrain.

<!-- ML4T_ZH_BEGIN c0008-p000 -->
## 2. 期間，以及允許查看該期間的模型

保留測試期間不是在此處決定的。它由 `evaluation.holdout_start` 與 `evaluation.holdout_end` 定義，兩者取自本案例研究自身的 `setup.yaml`，並透過推導驗證折與切取回測期間共同使用的 `canonical_window` 讀取，因此三者不會不一致。

訓練區間涵蓋該期間之前所有可用資料，並以標籤緩衝區作為上界。**對本案例研究而言，這個緩衝區確實發揮作用。** 標籤是未來 21 天報酬率，因此日期為 `t` 的資料列，所記錄的結果要等到 `t + 21` 個交易時段後才會得知；若訓練集一直延伸到保留測試期間開始當天，模型就會使用在保留測試期間內才確定報酬率的標籤進行擬合。因此，緩衝區必須涵蓋整個預測期間，並非例行設定；推導程序也不允許使用預設值，因為此處零間隔會造成資料洩漏。

組態其餘部分原樣沿用；無法直接沿用的欄位——適用資格清單，以及此模型家族依各驗證折自身訓練資料列決定的參數——則會針對保留測試折重新計算。若直接沿用，就會擬合一個仍以驗證折為識別依據的模型，卻稱它為重新訓練。
<!-- ML4T_ZH_END c0008-p000 -->


In [ ]:
observation_timeline = (
    pl.read_parquet(study.root / "labels" / f"{carrier['label']}.parquet")
    .get_column("timestamp")
    .unique()
    .sort()
    .to_list()
)
validation_spec = study.results.open(carrier["training_hash"]).spec()
holdout_spec = build_holdout_training_spec(
    study,
    validation_spec,
    timeline=observation_timeline,
    case_study=CASE_STUDY_ID,
)

fold = holdout_spec["computation"]["cv"]["folds"][0]
print(f"Holdout fold {fold['fold']}")
print(f"  trains  {fold['train_start']} -> {fold['train_end']}")
print(f"  predicts {fold['val_start']} -> {fold['val_end']}")
print(f"  label buffer: {holdout_spec['computation']['cv']['request']['label_buffer']}")

# The validation folds are what the buffer is measured against, and the last of them ends
# before the holdout opens. Printing both is what lets a reader check the gap rather than take
# it on the derivation's word.
validation_folds = validation_spec["computation"]["cv"]["folds"]
latest_validation_end = max(str(entry["val_end"]) for entry in validation_folds)
print(f"Validation folds: {len(validation_folds)}, latest evaluation end {latest_validation_end}")
print(f"Holdout training ends {fold['train_end']}, holdout opens {fold['val_start']}")

## 3. Fit, and register the predictions

`reconstruct_locked_model_request` builds the request from the spec above. Its name comes
from a locked holdout path this case study does not use; it takes a training specification
and a checkpoint, not a lock, and it is used here because it is the one call that refuses a
request that is not exactly the spec it was handed - the training identity, the checkpoint
schedule, the feature lineage and the runtime parameters are all checked before anything is
fitted.

The training identity below is new. It has to be: it covers the CV interval, and the holdout
fold is not one of the validation folds. A run that came back with the validation training
hash would mean the refit did not happen, so that is checked rather than assumed.

**The window carries one configuration, and this notebook has no way past that.** The check below
is on the selected configuration rather than on the notebook, and it has exactly two outcomes.
With the selected configuration unchanged this is an idempotent replay: the derivation is
deterministic and the training identity covers it, so the same identity comes back and the fit is
served from the registry, which is why re-running the notebook is free and safe. With the
selected configuration changed it refuses, names both configurations, and stops.


<!-- ML4T_ZH_BEGIN c0010-p000 -->
`reconstruct_locked_model_request` 會依上述規格建立請求。其名稱源自本案例研究未使用的鎖定保留測試路徑；它接收訓練規格與檢查點，而非鎖定資訊。這裡使用它，是因為只有這個呼叫會拒絕與所收到規格不完全相符的請求：在擬合任何模型前，它會檢查訓練識別項、檢查點時程、特徵沿革及執行參數。

下方的訓練識別項是新的，而且必須如此：它涵蓋交叉驗證區間，而保留測試折不屬於任何驗證折。若某次執行傳回驗證階段的訓練雜湊值，就代表沒有重新擬合；因此這裡會實際檢查，而不是直接假定。

**保留測試期間只對應一個組態，本 Notebook 沒有繞過此限制的方法。** 下方檢查是針對選定組態，而非 Notebook 本身，且只有兩種結果。若選定組態不變，這就是冪等的重播：推導程序具決定性，訓練識別項涵蓋其結果，因此會傳回相同的識別項，擬合結果也會從登錄資料提供；所以重新執行 Notebook 不增加成本且安全。若選定組態改變，檢查會拒絕執行、指出兩個組態並停止。
<!-- ML4T_ZH_END c0010-p000 -->

It refuses rather than offering a replacement switch, and the reason is that a replacement
would not be one. Deleting the earlier generation's rows does not undo having observed its
result: the selection that produced the new configuration may have been informed by the old
holdout number, and no deletion reaches that. A switch here would let the case study take a
second look at the window while leaving a registry that shows only one, which is the
specific thing that would make the out-of-sample claim false rather than merely weak.

<!-- ML4T_ZH_BEGIN c0010-p001 -->
它會拒絕執行，而不是提供替換開關，因為替換並不能抹去先前的觀察。刪除前一代的資料列，不等於沒有看過其結果：產生新組態的選擇可能受到舊保留測試數值影響，而刪除任何資料列都無法消除這項影響。若此處提供開關，案例研究便能再次查看保留測試期間，但登錄資料卻只顯示查看過一次；這正是會使樣本外主張失實，而不只是說服力變弱的情況。
<!-- ML4T_ZH_END c0010-p001 -->


In [ ]:
holdout_training_hash = training_hash_from_spec(holdout_spec)
this_generation = (holdout_training_hash, (CHECKPOINT_KIND, CHECKPOINT_VALUE))
retire = holdout_generations_to_retire(CASE_DIR, this_generation=this_generation)
# A row whose training run records no CV split cannot be shown either way, and deleting on
# that would discard a result nothing has established is wrong. It stops the run instead.
if retire.unattributable:
    raise RuntimeError(
        "the holdout window carries prediction sets whose training runs record no CV split, "
        "so whether they were refitted for the holdout cannot be established: "
        + ", ".join(
            f"{row['prediction_hash']} (training {row['training_hash']})"
            for row in retire.unattributable
        )
        + ". Establish what produced them before registering another evaluation on the same "
        "window; this notebook will not delete a row it cannot show is not a holdout result."
    )
# A row whose training run declares a non-holdout CV may not be reported as a holdout
# result, and it is also not something to delete unattended: `generate_holdout` refits on a
# holdout fold and then registers the predictions under the VALIDATION training identity, so
# this record covers both a validation-fitted model published over the window and a real
# refit filed under the wrong identity. Nothing owned this before - the filter here was
# `row["refitted"]`, which made exactly these rows invisible to the refusal and to
# everything after it.
if retire.not_out_of_sample:
    raise RuntimeError(
        "the holdout window carries prediction sets whose training runs declare a CV split "
        "other than the holdout: "
        + ", ".join(
            f"{row['prediction_hash']} ({row['config_name']}, training {row['training_hash']})"
            for row in retire.not_out_of_sample
        )
        + ". Each is either a validation-fitted model published over the window, which is "
        "not an out-of-sample result, or a refit registered under its validation training "
        "identity, which the retired `20_strategy_synthesis/holdout.py::generate_holdout` "
        "wrote until it was deleted on 2026-09-12 - and "
        "the registry cannot tell those apart. This notebook has no way past that: "
        "establish which it is and resolve it through the registry's own lifecycle, which "
        "records that the row was retired."
    )
superseded = list(retire.superseded)
if superseded:
    raise RuntimeError(
        "the holdout window already carries a refit of a different configuration: "
        + ", ".join(
            f"{row['prediction_hash']} ({row['config_name']}, training {row['training_hash']})"
            for row in superseded
        )
        + f". This run would evaluate {carrier['config_name']} (training "
        f"{holdout_training_hash}, checkpoint {CHECKPOINT_KIND}={CHECKPOINT_VALUE}) on the "
        "same window, which would be a second configuration measured on a period this case "
        "study reports as unseen. This notebook has no way past that: deleting the earlier "
        "generation would not undo having observed it, and the selection bias it introduces "
        "is not removed by removing the rows. Either leave the selection where it was, or "
        "retire the earlier evaluation through the registry's own lifecycle, which records "
        "that a second look was taken."
    )

In [ ]:
request = reconstruct_locked_model_request(
    study,
    holdout_spec,
    checkpoint_kind=CHECKPOINT_KIND,
    checkpoint_value=CHECKPOINT_VALUE,
)
model_run = request.run()
holdout_prediction = model_run.predictions[0]

if model_run.training.hash == carrier["training_hash"]:
    raise RuntimeError(
        "the holdout refit produced the validation training identity "
        f"{carrier['training_hash']}, which means it did not refit"
    )
print(f"Holdout training run:  {model_run.training.hash}")
print(f"Holdout prediction set: {holdout_prediction.hash}")

What the prediction set covers, read back from the registry rather than from the request.
The two agree only if the fit published what it declared, and the counts are what a reader
checks the window against: this is a daily panel, so the date count is trading sessions in
the window and the row count is those sessions times the funds eligible on each.

<!-- ML4T_ZH_BEGIN c0013-p000 -->
預測組合涵蓋的範圍，是從登錄資料讀回，而非直接取自請求。只有擬合程序確實發布了宣告內容時，兩者才會一致；讀者會用計數核對保留測試期間：這是每日資料面板，因此日期數是該期間的交易時段數，而資料列數等於各時段符合資格的基金數加總。
<!-- ML4T_ZH_END c0013-p000 -->


In [ ]:
record = holdout_prediction.registry_record()
predictions = holdout_prediction.load()
print(
    f"split={record['split']}  checkpoint={record['checkpoint_kind']}={record['checkpoint_value']}"
)
print(f"rows={predictions.height:,}  sessions={predictions['timestamp'].n_unique():,}")
print(
    f"  {predictions['timestamp'].min()} -> {predictions['timestamp'].max()}, "
    f"{predictions['symbol'].n_unique():,} funds"
)

Every holdout prediction set the registry holds, and whether the model behind it was
refitted for the window. All of them are listed rather than one silently preferred, because
the registry is immutable and a reader looking at it later will see whatever is there. A row
marked VALIDATION-FITTED is not an out-of-sample result whatever its numbers say.

<!-- ML4T_ZH_BEGIN c0015-p000 -->
列出登錄資料持有的每一組保留測試集預測，以及其背後的模型是否針對該期間重新擬合。所有預測組合都會列出，而非悄悄偏好其中一組，因為登錄資料不可變，日後查看的讀者會看到其中實際存在的所有內容。標記為 VALIDATION-FITTED 的資料列，無論數值如何，都不是樣本外結果。
<!-- ML4T_ZH_END c0015-p000 -->


In [ ]:
for row in registered_holdout_generations(CASE_DIR):
    note = (
        "refitted for the holdout" if row["refitted"] else "VALIDATION-FITTED - not out of sample"
    )
    print(
        f"  {row['prediction_hash']}  training={row['training_hash']}  {row['config_name']}  {note}"
    )

## What this notebook establishes, and what it does not

It establishes one thing: a prediction set over the holdout window, produced by the
configuration this case study selected, fitted on data that ends a full label horizon before
the window opens. That is a precondition for an out-of-sample claim, not the claim itself.
Nothing here says whether the predictions are any good - they have not been scored, sized or
traded.

It does not make the holdout a fresh test in the strict sense. The configuration reached this
notebook through a selection made on the validation folds, and this window is being used once
per configuration that gets here. What it does remove is the specific circularity of scoring
a validation-fitted model on the period meant to judge it.

Re-running this notebook is free: the same configuration re-derives the same training identity and
the fit is served from the registry. Evaluating a DIFFERENT configuration is not, and is
refused here. If a later pass finds the selection was wrong, that is a question for the
registry's lifecycle, which records that a second look was taken - not something to settle by
deleting rows until the registry agrees.

**Next:** [`19_holdout_backtest`](19_holdout_backtest.ipynb).

<!-- ML4T_ZH_BEGIN c0017-p000 -->
## 本 Notebook 確立了什麼，以及尚未確立什麼

它確立一件事：由本案例研究選出的組態，在保留測試期間產生一組預測，而模型擬合所用的資料會在該期間開始前，提早整個標籤預測期間結束。這是提出樣本外主張的先決條件，還不是主張本身。此處尚無法判定預測是否良好，因為尚未評分、設定部位規模或交易。

嚴格來說，它也無法讓保留測試集變成全新的測試。組態是透過驗證折上的篩選來到這個 Notebook，而每個抵達此處的組態都會使用這段期間一次。它確實排除的是一種特定的循環論證：拿在驗證期間擬合的模型，去評分原本要用來檢驗它的期間。

重新執行本 Notebook 不增加成本：同一組態會重新推導出相同的訓練識別項，擬合結果由登錄資料提供。但評估**不同**組態就不是如此，因此會在此處遭到拒絕。若日後發現原先選擇有誤，那應交由登錄資料的生命週期處理，明確記錄曾經再次查看保留測試集，而不是反覆刪除資料列，直到登錄資料符合預期。

**下一步：**[`19_holdout_backtest`](19_holdout_backtest.ipynb)。
<!-- ML4T_ZH_END c0017-p000 -->
